# 🥉 Bronze Layer — Demonstração

**Tech Challenge – Fase 2 | PosTech FIAP**

Este notebook demonstra **passo a passo** o que o script `pipeline/batch/bronze/bronze_loader.py` faz.

### Regras da camada Bronze
- ✅ Dados preservados **exatamente** como vieram da fonte
- ✅ Apenas adição de **metadados de ingestão**
- ✅ Gravação em **Parquet** para eficiência
- ❌ Nenhuma transformação de negócio
- ❌ Sem limpeza de nulos ou mudança de tipos

In [1]:
import pandas as pd
import os
from pathlib import Path
from datetime import datetime, timezone

RAW_DIR    = Path("..") / "data" / "raw"
BRONZE_DIR = Path("..") / "layers" / "bronze"

PIPELINE_VERSION = "1.0.0"

print("Caminhos configurados.")
print(f"  Origem:  {RAW_DIR.resolve()}")
print(f"  Destino: {BRONZE_DIR.resolve()}")

Caminhos configurados.
  Origem:  /home/mateus-kent/projects/1IAST-Tech-Challenge-Fase2/data/raw
  Destino: /home/mateus-kent/projects/1IAST-Tech-Challenge-Fase2/layers/bronze


## Passo 1 — Leitura do CSV bruto

Usamos `dtype=str` para preservar tudo exatamente como está na fonte — sem conversões automáticas de tipo.

In [2]:
# Vamos usar indicador_municipio como exemplo (maior tabela)
csv_path = RAW_DIR / "indicador_municipio.csv"

df_raw = pd.read_csv(csv_path, dtype=str)

print(f"Shape: {df_raw.shape}")
print(f"Colunas: {list(df_raw.columns)}")
df_raw.head()

Shape: (23995, 15)
Colunas: ['ano', 'id_municipio', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8']


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,1100031,2,3,69.1,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1100072,2,3,58.2,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,1101609,2,3,50.7,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Todos os tipos são string — intencional na Bronze
df_raw.dtypes

ano                        str
id_municipio               str
serie                      str
rede                       str
taxa_alfabetizacao         str
media_portugues            str
proporcao_aluno_nivel_0    str
proporcao_aluno_nivel_1    str
proporcao_aluno_nivel_2    str
proporcao_aluno_nivel_3    str
proporcao_aluno_nivel_4    str
proporcao_aluno_nivel_5    str
proporcao_aluno_nivel_6    str
proporcao_aluno_nivel_7    str
proporcao_aluno_nivel_8    str
dtype: object

## Passo 2 — Adição de metadados de ingestão

Adicionamos 3 colunas de rastreabilidade a cada linha. Isso é fundamental para:
- Saber **quando** o dado foi ingerido
- Saber **de onde** ele veio
- Rastrear **qual versão** do pipeline gerou o arquivo

In [4]:
df_bronze = df_raw.copy()

df_bronze["_ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
df_bronze["_source_file"]         = "indicador_municipio.csv"
df_bronze["_pipeline_version"]    = PIPELINE_VERSION

print("Colunas após adição de metadados:")
print(list(df_bronze.columns))

Colunas após adição de metadados:
['ano', 'id_municipio', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', '_ingestion_timestamp', '_source_file', '_pipeline_version']


In [5]:
# Visualiza as colunas de metadados
df_bronze[["ano", "id_municipio", "taxa_alfabetizacao",
           "_ingestion_timestamp", "_source_file", "_pipeline_version"]].head()

,ano,id_municipio,taxa_alfabetizacao,_ingestion_timestamp,_source_file,_pipeline_version
0,2023,1100031,69.1,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
1,2023,1100072,58.2,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
2,2023,1100189,69.73,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
3,2023,1101609,50.7,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
4,2023,1101807,55.69,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0


## Passo 3 — Gravação em Parquet

In [6]:
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = BRONZE_DIR / "indicador_municipio.parquet"

df_bronze.to_parquet(parquet_path, index=False, engine="pyarrow")

size_csv_kb    = csv_path.stat().st_size / 1024
size_parquet_kb = parquet_path.stat().st_size / 1024
reducao_pct    = (1 - size_parquet_kb / size_csv_kb) * 100

print(f"CSV original:       {size_csv_kb:>8.1f} KB")
print(f"Parquet gerado:     {size_parquet_kb:>8.1f} KB")
print(f"Redução de tamanho: {reducao_pct:>8.1f}%")

CSV original:         1362.3 KB
Parquet gerado:        491.8 KB
Redução de tamanho:     63.9%


## Passo 4 — Verificação do arquivo gerado

In [7]:
# Lê o Parquet para confirmar integridade
df_verificado = pd.read_parquet(parquet_path)

print(f"Shape do Parquet lido: {df_verificado.shape}")
print(f"Registros preservados: {len(df_raw) == len(df_verificado)}")
df_verificado.head()

Shape do Parquet lido: (23995, 18)
Registros preservados: True


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_file,_pipeline_version
0,2023,1100031,2,3,69.1,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
1,2023,1100072,2,3,58.2,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
3,2023,1101609,2,3,50.7,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26T23:48:26.801014+00:00,indicador_municipio.csv,1.0.0


## Passo 5 — Rodando o pipeline completo (todos os arquivos)

In [8]:
import sys
sys.path.append(str(Path("..")))

from pipeline.batch.bronze.bronze_loader import run

resultados = run()

2026-05-26 20:48:26 [INFO] ============================================================


2026-05-26 20:48:26 [INFO] Bronze Layer — início da ingestão batch


2026-05-26 20:48:26 [INFO] Versão do pipeline: 1.0.0


2026-05-26 20:48:26 [INFO] ============================================================


2026-05-26 20:48:26 [INFO] Iniciando ingestão: meta_brasil ← meta_alfabetizacao_brasil.csv


...

2026-05-26 20:48:26 [INFO]   ✔ indicador_uf              145 registros  (17.2 KB)


2026-05-26 20:48:26 [INFO] ------------------------------------------------------------


2026-05-26 20:48:26 [INFO]   Total ingerido: 34901 registros


2026-05-26 20:48:26 [INFO]   Destino:        /home/mateus-kent/projects/1IAST-Tech-Challenge-Fase2/layers/bronze


2026-05-26 20:48:26 [INFO] ============================================================


## Passo 6 — Resumo da camada Bronze

In [9]:
# Lista todos os Parquets gerados com tamanho
print(f"{'Arquivo':<35} {'Tamanho':>10} {'Registros':>12}")
print("-" * 60)

total_registros = 0
total_kb = 0

for r in resultados:
    if r["status"] == "ok":
        print(f"{r['arquivo_parquet']:<35} {r['tamanho_kb']:>8.1f} KB {r['registros']:>10,}")
        total_registros += r["registros"]
        total_kb += r["tamanho_kb"]

print("-" * 60)
print(f"{'TOTAL':<35} {total_kb:>8.1f} KB {total_registros:>10,}")

Arquivo                                Tamanho    Registros
------------------------------------------------------------
meta_brasil.parquet                      8.9 KB          3
meta_municipio.parquet                 214.1 KB     10,704
meta_uf.parquet                         11.1 KB         54
indicador_municipio.parquet            491.8 KB     23,995
indicador_uf.parquet                    17.2 KB        145
------------------------------------------------------------
TOTAL                                  743.1 KB     34,901


## Conclusão

A camada **Bronze** está concluída. Os dados foram:

| Etapa | Status |
|---|---|
| Lidos dos CSVs originais | ✅ |
| Preservados sem transformações | ✅ |
| Metadados de ingestão adicionados | ✅ |
| Gravados em Parquet | ✅ |

**Próximo passo:** Silver Layer — limpeza, tipagem correta, tratamento de nulos e JOIN entre as tabelas.